<a href="https://colab.research.google.com/github/Zuhair0000/tensorflow_bootcamp/blob/main/08_intoduction_to_NLP_in_tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py

--2026-02-28 15:20:13--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0.001s  

2026-02-28 15:20:14 (17.3 MB/s) - ‘helper_functions.py’ saved [10246/10246]



In [2]:
from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

In [3]:
!wget https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
unzip_data("nlp_getting_started.zip")

--2026-02-28 15:20:38--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 172.217.203.207, 108.177.11.207, 142.250.98.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.217.203.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip’

nlp_getting_started 100%[===================>] 593.11K  --.-KB/s    in 0.01s   

2026-02-28 15:20:38 (59.6 MB/s) - ‘nlp_getting_started.zip’ saved [607343/607343]



# Visualizing a text dataset

In [4]:
import pandas as pd
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [5]:
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [6]:
train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


In [7]:
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [8]:
train_df_shuffled.target.value_counts()

,count
target,
0,4342
1,3271


In [9]:
len(train_df_shuffled)

7613

In [10]:
len(test_df)

3263

In [11]:
import random
random_index = random.randint(0, len(train_df)-5)
for row in train_df_shuffled[['text', 'target']][random_index: random_index+5].itertuples():
  _, text, target = row
  print(f"Target: {target}", "(real disaster)" if target > 0 else "(not real disaster)")
  print(f"Text:\n{text}\n")
  print("---\n")

Target: 1 (real disaster)
Text:
Canada: Hailstorm flash flooding slam Calgary knocks out power to 20k customers
http://t.co/SkY9EokgGB http://t.co/5IyZsDA6xB

---

Target: 0 (not real disaster)
Text:
This bowl got me thinking... Damn I've been blazing for so damn long

---

Target: 0 (not real disaster)
Text:
The way you move is like a full on rainstorm and I'm a house of cards.

---

Target: 1 (real disaster)
Text:
Newlyweds feed thousands of Syrian refugees instead of hosting a banquet wedding dinner -  http://t.co/XZV0lT9ZZk via @smh

---

Target: 0 (not real disaster)
Text:
Is This Country Latin America's Next 'Argentina': One week ago we reported on the economic devastation in he o... http://t.co/m2y9Ym3iF6

---



# Split Data

In [12]:
from sklearn.model_selection import train_test_split
train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled['text'].to_numpy(),
                                                                            train_df_shuffled['target'].to_numpy(),
                                                                            test_size=0.1,
                                                                            random_state=42)


In [13]:
len(train_sentences), len(train_labels), len(val_sentences), len(val_labels)

(6851, 6851, 762, 762)

# Converting text to numbers

In [14]:
train_sentences[:5]

array(['@mogacola @zamtriossu i screamed after hitting tweet',
       'Imagine getting flattened by Kurt Zouma',
       '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
       "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
       'Somehow find you and I collide http://t.co/Ee8RpOahPk'],
      dtype=object)

In [15]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

text_vectorizer = TextVectorization(max_tokens=None,
                                    standardize='lower_and_strip_punctuation',
                                    split="whitespace",
                                    ngrams=None,
                                    output_mode='int',
                                    output_sequence_length=None,
                                    pad_to_max_tokens=False
                                   )

In [16]:
round(sum([len(i.split()) for i in train_sentences])/len(train_sentences))

15

In [17]:
max_vocab_length = 10000
max_length = 15

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode = 'int',
                                    output_sequence_length=max_length)

In [18]:
text_vectorizer.adapt(train_sentences)

In [19]:
sample_sentence = "There's a flood in my street!"
text_vectorizer([sample_sentence])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[264,   3, 232,   4,  13, 698,   0,   0,   0,   0,   0,   0,   0,
          0,   0]])>

In [20]:
random_sentence = random.choice(train_sentences)
print(f" Original text:\n {random_sentence} \
        \n\nVectorized version:")
text_vectorizer([random_sentence])

 Original text:
 @alextucker VOLCANO BOWL DRINK         

Vectorized version:


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[   1,  524, 2536, 1530,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0]])>

In [21]:
words_in_vocab = text_vectorizer.get_vocabulary()
top_5_words = words_in_vocab[:5]
bottom_5_words = words_in_vocab[-5:]

print(f"Number of words in vocab: {len(words_in_vocab)}")
print(f"5 most common words: {top_5_words}")
print(f"5 least common words: {bottom_5_words}")

Number of words in vocab: 10000
5 most common words: ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
5 least common words: [np.str_('pages'), np.str_('paeds'), np.str_('pads'), np.str_('padres'), np.str_('paddytomlinson1')]


# Creating an embedding layer

In [22]:
embedding = tf.keras.layers.Embedding(input_dim=max_vocab_length,
                                      output_dim=128,
                                      input_length=max_length
                                      )

embedding

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


<Embedding name=embedding, built=False>

In [23]:
random_sentences = random.choice(train_sentences)
print(f"Original text:\n {random_sentence}\
        \n\nEmbedded version:")
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed

Original text:
 @alextucker VOLCANO BOWL DRINK        

Embedded version:


<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[-0.005811  , -0.0050388 ,  0.00956219, ..., -0.04200827,
          0.01109425,  0.00881491],
        [-0.01398944, -0.03104429,  0.0496656 , ..., -0.04737382,
          0.03534186,  0.03126163],
        [-0.02166023, -0.04701481,  0.01368573, ...,  0.02632245,
         -0.04474827, -0.02311999],
        ...,
        [ 0.03978998, -0.03705864,  0.01970645, ...,  0.04284178,
         -0.03790962, -0.02690381],
        [ 0.03978998, -0.03705864,  0.01970645, ...,  0.04284178,
         -0.03790962, -0.02690381],
        [ 0.03978998, -0.03705864,  0.01970645, ...,  0.04284178,
         -0.03790962, -0.02690381]]], dtype=float32)>

## Modelling a text dataset

### Model 0

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

model_0 = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('model', MultinomialNB())
])

model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('model', MultinomialNB())])

In [25]:
baseline_score = model_0.score(val_sentences, val_labels)
print(f"Baseline model acieves an accuracy of: {baseline_score*100:.2f}")

Baseline model acieves an accuracy of: 79.27


In [26]:
baseline_prediction = model_0.predict(val_sentences)

In [27]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate(y_pred, y_true):
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
  return{
      "Accuracy Score": accuracy_score(y_true, y_pred),
      "Model Precision": model_precision,
      "Recall": model_recall,
      "F1": model_f1
  }

In [28]:
baseline_results = evaluate(val_labels, baseline_prediction)
baseline_results

{'Accuracy Score': 0.7926509186351706,
 'Model Precision': 0.8336022277575122,
 'Recall': 0.7926509186351706,
 'F1': 0.7990828614653861}

### Model 1

In [29]:
from helper_functions import create_tensorboard_callback

SAVE_DIR = 'model_logs'

In [30]:
inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)
x = text_vectorizer(inputs)
x = embedding(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)


model_1 = tf.keras.Model(inputs, outputs, name='model_1_dense')

In [31]:
model_1.compile(loss='binary_crossentropy',
                optimizer='Adam',
                metrics=['accuracy'])

In [32]:
history_1 = model_1.fit(x = train_sentences,
                        y=train_labels,
                        epochs=5,
                        validation_data = (val_sentences, val_labels),
                        callbacks=[create_tensorboard_callback(dir_name='SAVE_DIR',
                                                               experiment_name='model_1_dense')])

Saving TensorBoard log files to: SAVE_DIR/model_1_dense/20260228-152040
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.6479 - loss: 0.6487 - val_accuracy: 0.7428 - val_loss: 0.5415
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.8127 - loss: 0.4563 - val_accuracy: 0.7848 - val_loss: 0.4696
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.8537 - loss: 0.3601 - val_accuracy: 0.7861 - val_loss: 0.4622
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.8913 - loss: 0.2901 - val_accuracy: 0.7835 - val_loss: 0.4615
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.9148 - loss: 0.2393 - val_accuracy: 0.7913 - val_loss: 0.4832


In [33]:
model_1.evaluate(val_sentences, val_labels)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7844 - loss: 0.5179


[0.48318424820899963, 0.7913385629653931]

In [34]:
model_1_pred_probs = model_1.predict(val_sentences)
model_1_pred_probs.shape

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


(762, 1)

In [35]:
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs))
model_1_preds[:20]

<tf.Tensor: shape=(20,), dtype=float32, numpy=
array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0.], dtype=float32)>

In [36]:
model_1_results = evaluate(val_labels,
                           model_1_preds)
model_1_results, baseline_results

({'Accuracy Score': 0.7913385826771654,
  'Model Precision': 0.8133217949205624,
  'Recall': 0.7913385826771654,
  'F1': 0.7951105417837312},
 {'Accuracy Score': 0.7926509186351706,
  'Model Precision': 0.8336022277575122,
  'Recall': 0.7926509186351706,
  'F1': 0.7990828614653861})

# Visualizing Embeddings

In [37]:
words_in_vocab = text_vectorizer.get_vocabulary()
len(words_in_vocab), words_in_vocab[:10]

(10000,
 ['',
  '[UNK]',
  np.str_('the'),
  np.str_('a'),
  np.str_('in'),
  np.str_('to'),
  np.str_('of'),
  np.str_('and'),
  np.str_('i'),
  np.str_('is')])

In [38]:
model_1.summary()

Model: "model_1_dense"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,840,389 (14.65 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,560,260 (9.77 MB)

In [39]:
embed_weights = model_1.get_layer('embedding').get_weights()[0]
print(embed_weights.shape)

(10000, 128)


In [40]:
import io
out_v = io.open("vectors.tsv", 'w', encoding='utf-8')
out_m = io.open("metadata.tsv", 'w', encoding='utf-8')

for index, word in enumerate(words_in_vocab):
  if index == 0:
    continue
  vec = embed_weights[index]
  out_v.write('\t'.join([str(x) for x in vec]) + '\n')
  out_m.write(word + '\n')
out_v.close()
out_m.close()

In [41]:
try:
  from google.colab import files
  files.download('vectors.tsv')
  files.download('metadata.tsv')
except:
  pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# RNN

## LSTM

In [42]:
inputs = tf.keras.layers.Input(shape=(1,), dtype='string')
x = text_vectorizer(inputs)
x = embedding(x)
x = tf.keras.layers.LSTM(64, return_sequences=True)(x)
x = tf.keras.layers.LSTM(64)(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model_2 = tf.keras.Model(inputs, outputs, name='model_2_LSTM')

In [43]:
model_2.compile(loss='binary_crossentropy',
                optimizer='Adam',
                metrics=['accuracy'])

In [44]:
history_2 = model_2.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences, val_labels),
                        callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                                               experiment_name='model_2_LSTM')])

Saving TensorBoard log files to: model_logs/model_2_LSTM/20260228-152105
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 14s 44ms/step - accuracy: 0.8985 - loss: 0.2996 - val_accuracy: 0.7625 - val_loss: 0.5831
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.9419 - loss: 0.1560 - val_accuracy: 0.7795 - val_loss: 0.5702
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.9501 - loss: 0.1319 - val_accuracy: 0.7848 - val_loss: 0.8664
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 38ms/step - accuracy: 0.9607 - loss: 0.1002 - val_accuracy: 0.7808 - val_loss: 0.7682
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 38ms/step - accuracy: 0.9653 - loss: 0.0844 - val_accuracy: 0.7651 - val_loss: 1.0561


In [45]:
model_2_pred_probs = model_2.predict(val_sentences)
model_2_pred_probs[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step


array([[1.7395552e-03],
       [5.5462295e-01],
       [9.9998283e-01],
       [7.9261191e-02],
       [3.7720671e-04],
       [9.9989039e-01],
       [9.4270092e-01],
       [9.9998879e-01],
       [9.9998552e-01],
       [3.8163549e-01]], dtype=float32)

In [46]:
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))
model_2_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [47]:
model_2_results = evaluate(val_labels,
                           model_2_preds)
model_2_results

{'Accuracy Score': 0.7650918635170604,
 'Model Precision': 0.778549428260498,
 'Recall': 0.7650918635170604,
 'F1': 0.7678538871252285}

In [48]:
baseline_results

{'Accuracy Score': 0.7926509186351706,
 'Model Precision': 0.8336022277575122,
 'Recall': 0.7926509186351706,
 'F1': 0.7990828614653861}

## GRU

In [49]:
inputs = tf.keras.layers.Input(shape=(1,), dtype='string')
x = text_vectorizer(inputs)
x = embedding(x)
x = tf.keras.layers.GRU(64, return_sequences=True)(x)
x = tf.keras.layers.LSTM(64, return_sequences=True)(x)
x = tf.keras.layers.GRU(64)(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model_3 = tf.keras.Model(inputs, outputs, name='model_3_GRU')

In [50]:
model_3.compile(loss='binary_crossentropy',
                optimizer='Adam',
                metrics=['accuracy'])

In [51]:
history_3 = model_3.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences, val_labels),
                        callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                                               experiment_name='model_3_GRU')])

Saving TensorBoard log files to: model_logs/model_3_GRU/20260228-152156
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 20s 63ms/step - accuracy: 0.9230 - loss: 0.2319 - val_accuracy: 0.7795 - val_loss: 0.7715
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 18s 50ms/step - accuracy: 0.9693 - loss: 0.0783 - val_accuracy: 0.7730 - val_loss: 1.1309
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - accuracy: 0.9743 - loss: 0.0618 - val_accuracy: 0.7730 - val_loss: 1.1393
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 10s 48ms/step - accuracy: 0.9779 - loss: 0.0526 - val_accuracy: 0.7690 - val_loss: 1.2791
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 10s 49ms/step - accuracy: 0.9794 - loss: 0.0485 - val_accuracy: 0.7677 - val_loss: 1.5740


In [52]:
model_3_pred_probs = model_3.predict(val_sentences)
model_3_pred_probs[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step


array([[9.3329705e-05],
       [7.2174168e-01],
       [9.9989432e-01],
       [1.1677934e-03],
       [9.4717070e-06],
       [9.9923927e-01],
       [9.7755814e-01],
       [9.9989069e-01],
       [9.9986333e-01],
       [5.8504885e-01]], dtype=float32)

In [53]:
model_3_preds = tf.squeeze(tf.round(model_3_pred_probs))
model_3_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [54]:
model_3_results = evaluate(val_labels,
                           model_3_preds)
model_3_results

{'Accuracy Score': 0.7677165354330708,
 'Model Precision': 0.7757318300604729,
 'Recall': 0.7677165354330708,
 'F1': 0.769506831481095}

# Bidirectional LSTM

In [55]:
inputs = tf.keras.layers.Input(shape=(1,), dtype='string')
x = text_vectorizer(inputs)
x = embedding(x)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(x)
x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64))(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model_4 = tf.keras.Model(inputs, outputs)

In [56]:
model_4.compile(loss='binary_crossentropy',
                optimizer='Adam',
                metrics=['accuracy'])

In [57]:
history_4 = model_4.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences, val_labels),
                        callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                                               experiment_name='model_4_Bidirectional')])

Saving TensorBoard log files to: model_logs/model_4_Bidirectional/20260228-152315
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 22s 65ms/step - accuracy: 0.9372 - loss: 0.1823 - val_accuracy: 0.7730 - val_loss: 1.1604
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 19s 59ms/step - accuracy: 0.9744 - loss: 0.0546 - val_accuracy: 0.7677 - val_loss: 1.2905
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 13s 60ms/step - accuracy: 0.9806 - loss: 0.0450 - val_accuracy: 0.7743 - val_loss: 1.1695
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 20s 59ms/step - accuracy: 0.9788 - loss: 0.0435 - val_accuracy: 0.7703 - val_loss: 1.1711
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 12s 58ms/step - accuracy: 0.9812 - loss: 0.0370 - val_accuracy: 0.7612 - val_loss: 1.3722


In [58]:
model_4_pred_probs = model_4.predict(val_sentences)
model_4_pred_probs[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step


array([[2.4309471e-02],
       [7.7449405e-01],
       [9.9997061e-01],
       [5.0194383e-01],
       [3.2994634e-05],
       [9.9992359e-01],
       [9.9623072e-01],
       [9.9997783e-01],
       [9.9996841e-01],
       [5.4662991e-01]], dtype=float32)

In [59]:
model_4_preds = tf.squeeze(tf.round(model_4_pred_probs))
model_4_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 1., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [60]:
model_4_results = evaluate(val_labels,
                           model_4_preds)
model_4_results

{'Accuracy Score': 0.7611548556430446,
 'Model Precision': 0.7618146307423717,
 'Recall': 0.7611548556430446,
 'F1': 0.7614006904694942}

# Convolutional Nueral Netowrks for Text

In [61]:
embedding_test = embedding(text_vectorizer(['this is a test sentence']))
conv_1d = tf.keras.layers.Conv1D(filters=32,
                                 kernel_size=5, # It means it lokks at 5 words at a time
                                 activation='relu',
                                 padding='valid', #default = 'valid', the output is smaller the the input shape, 'same' means output is
                                 )
conv_1d_output = conv_1d(embedding_test) # pass test embedding through conv1D layer
max_pool = tf.keras.layers.GlobalMaxPool1D()
max_pool_output = max_pool(conv_1d_output)

embedding_test.shape, conv_1d_output.shape, max_pool_output.shape

(TensorShape([1, 15, 128]), TensorShape([1, 11, 32]), TensorShape([1, 32]))

In [62]:
inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)
x = text_vectorizer(inputs)
x = embedding(x)
x = tf.keras.layers.Conv1D(64, 5, activation='relu', padding='valid')(x)
x = tf.keras.layers.GlobalMaxPool1D()(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model_5 = tf.keras.Model(inputs, outputs)

In [63]:
model_5.compile(loss='binary_crossentropy',
                optimizer='Adam',
                metrics=['accuracy'])

In [64]:
history_5 = model_5.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences, val_labels),
                        callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                                               experiment_name='Conv1D')])

Saving TensorBoard log files to: model_logs/Conv1D/20260228-152444
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.9429 - loss: 0.1770 - val_accuracy: 0.7677 - val_loss: 0.8982
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.9741 - loss: 0.0702 - val_accuracy: 0.7625 - val_loss: 1.0150
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.9751 - loss: 0.0597 - val_accuracy: 0.7664 - val_loss: 1.1026
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.9782 - loss: 0.0511 - val_accuracy: 0.7625 - val_loss: 1.1388
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.9793 - loss: 0.0498 - val_accuracy: 0.7598 - val_loss: 1.1990


In [65]:
model_5_pred_probs = model_5.predict(val_sentences)
model_5_pred_probs[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


array([[5.3069037e-01],
       [9.2742896e-01],
       [9.9994785e-01],
       [4.7706433e-02],
       [5.5732869e-07],
       [9.9901980e-01],
       [9.6526766e-01],
       [9.9997467e-01],
       [9.9999958e-01],
       [6.2933671e-01]], dtype=float32)

In [66]:
model_5_preds = tf.squeeze(tf.round(model_5_pred_probs))
model_5_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([1., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [67]:
model_5_results = evaluate(val_labels,
                           model_5_preds)
model_5_results

{'Accuracy Score': 0.7598425196850394,
 'Model Precision': 0.7617806363659814,
 'Recall': 0.7598425196850394,
 'F1': 0.7604331240453053}

# Model 6: Tensorflow hub for pretrained word embeddings

In [69]:
import tensorflow_hub as hub

embed = hub.load('https://www.kaggle.com/models/google/universal-sentence-encoder/TensorFlow2/universal-sentence-encoder/2')
embed_samples = embed([sample_sentence,
                       "When you can the universsal the sentence encoder on a sentence, it turns is into numbers"])

print(embed_samples[0][:50])

tf.Tensor(
[-0.01157025  0.02485911  0.02878051 -0.012715    0.03971541  0.08827761
  0.02680988  0.05589838 -0.01068731 -0.00597293  0.00639321 -0.01819516
  0.00030816  0.09105889  0.05874645 -0.03180629  0.01512474 -0.05162925
  0.00991366 -0.06865345 -0.04209306  0.0267898   0.03011009  0.00321065
 -0.00337968 -0.04787356  0.0226672  -0.00985927 -0.04063615 -0.01292093
 -0.04666382  0.05630299 -0.03949255  0.00517682  0.02495827 -0.07014439
  0.0287151   0.0494768  -0.00633978 -0.08960193  0.02807119 -0.00808364
 -0.01360601  0.05998649 -0.10361788 -0.05195372  0.00232958 -0.02332531
 -0.03758106  0.03327729], shape=(50,), dtype=float32)


In [73]:
# sentence_encoder_layer = hub.KerasLayer("https://www.kaggle.com/models/google/universal-sentence-encoder/TensorFlow2/universal-sentence-encoder/2",
#                                         input_shape=[],
#                                         dtype=tf.string,
#                                         trainable=False,
#                                         name='USE')

In [76]:
from keras.layers import Layer

class HubLayerWrapper(Layer):
    def __init__(self, hub_url, **kwargs):
        super().__init__(**kwargs)
        self.hub_layer = hub.KerasLayer(hub_url, trainable=False)

    def call(self, inputs):
        return self.hub_layer(inputs)

# 2. Use the wrapper in your Functional model
sentence_encoder_layer = HubLayerWrapper("https://www.kaggle.com/models/google/universal-sentence-encoder/TensorFlow2/universal-sentence-encoder/2")

In [77]:
inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
x = sentence_encoder_layer(inputs)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model_6 = tf.keras.Model(inputs, outputs)

In [78]:
model_6.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None)                 │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hub_layer_wrapper               │ (None, 512)            │             0 │
│ (HubLayerWrapper)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 513 (2.00 KB)

 Trainable params: 513 (2.00 KB)

 Non-trainable params: 0 (0.00 B)

In [80]:
model_6.compile(loss='binary_crossentropy',
                optimizer='Adam',
                metrics=['accuracy'])

In [81]:
history_6 = model_6.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences, val_labels),
                        callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                                               experiment_name='tf_hub_sentence_encoder')])

Saving TensorBoard log files to: model_logs/tf_hub_sentence_encoder/20260228-160706
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.6287 - loss: 0.6740 - val_accuracy: 0.7782 - val_loss: 0.6162
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.7876 - loss: 0.5948 - val_accuracy: 0.7808 - val_loss: 0.5661
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8028 - loss: 0.5453 - val_accuracy: 0.7795 - val_loss: 0.5340
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.7956 - loss: 0.5192 - val_accuracy: 0.7835 - val_loss: 0.5121
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8050 - loss: 0.4926 - val_accuracy: 0.7887 - val_loss: 0.4973


In [82]:
model_6_pred_probs = model_6.predict(val_sentences)
model_6_pred_probs[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step


array([[0.36499318],
       [0.6745543 ],
       [0.8503142 ],
       [0.3491037 ],
       [0.64627266],
       [0.7287929 ],
       [0.82585883],
       [0.83780736],
       [0.7503678 ],
       [0.19726026]], dtype=float32)

In [83]:
model_6_preds = tf.squeeze(tf.round(model_6_pred_probs))
model_6_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 1., 1., 1., 1., 1., 0.], dtype=float32)>

In [84]:
model_6_results = evaluate(val_labels,
                           model_6_preds)
model_6_results

{'Accuracy Score': 0.7887139107611548,
 'Model Precision': 0.7940903236046019,
 'Recall': 0.7887139107611548,
 'F1': 0.7899231252974518}